# Step 1 — Fine-tuned Baselines: BERT, HateBERT, and RoBERTa on IHC and ISHate

We fine-tune three pretrained models on two hate speech corpora (IHC and ISHate) and compare their macro F1 scores across both datasets.

In [1]:
# Uncomment if running in Colab
# !pip install -q transformers datasets accelerate scikit-learn

In [2]:
import os
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings("ignore")

## 1. Load the Datasets

### IHC (Implicit Hate Corpus)
`SALT-NLP/implicit-hate` was removed from the Hub — `tasksource/implicit-hate-stg1` is a verified mirror of the same IHC corpus (ElSherief et al., 2021). It has a single `train` split with 21,480 tweets labelled as `not_hate`, `explicit_hate`, or `implicit_hate`. Since there is no pre-made test split, we hold out 10% for evaluation. The binary label collapses explicit and implicit hate into a single **HS** class (label = 1); non-hate is **Non-HS** (label = 0).

### ISHate (Implicit and Subtle Hate Speech)
`BenjaminOcampo/ISHate` is a large-scale dataset (63,758 examples) focused on implicit and subtle hate speech, annotated with multiple layers. We use the `hateful_layer` column for binary classification (`Non-HS` → 0, `HS` → 1) and its pre-made train/test splits.

In [3]:
# IHC
raw = load_dataset("tasksource/implicit-hate-stg1", split="train")
splits = raw.train_test_split(test_size=0.10, seed=42)

def add_binary_label_ihc(example):
    example["label"] = 0 if example["class"] == "not_hate" else 1
    return example

train_ds = splits["train"].map(add_binary_label_ihc)
test_ds  = splits["test"].map(add_binary_label_ihc)

print("IHC")
print(f"  Train: {len(train_ds):,}  Test: {len(test_ds):,}")

# ISHate
ishate_raw = load_dataset("BenjaminOcampo/ISHate")

def add_binary_label_ishate(example):
    example["label"] = 0 if example["hateful_layer"] == "Non-HS" else 1
    return example

ishate_train = ishate_raw["train"].map(add_binary_label_ishate)
ishate_test  = ishate_raw["test"].map(add_binary_label_ishate)

print("\nISHate")
print(f"  Train: {len(ishate_train):,}  Test: {len(ishate_test):,}")

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]


ISHate
  Train: 55,023  Test: 4,368


### Vicomtech (Hate Speech Dataset — Stormfront)
`Vicomtech/hate-speech-dataset` (de Gibert et al., 2018) is a collection of sentences extracted from Stormfront white-nationalist forum posts. Each sentence is stored as an individual `.txt` file and labelled via `annotations_metadata.csv`. We use the pre-made `sampled_train` / `sampled_test` splits (balanced classes). The `idk/skip` and `relation` labels are discarded. The binary label maps `hate` → 1, `noHate` → 0.

In [4]:
import urllib.request
import zipfile
import shutil
import pandas as pd

# Download zip archive once; re-download if cache is incomplete/corrupted
_repo_dir = "./data/hate-speech-dataset"
_metadata_path = f"{_repo_dir}/annotations_metadata.csv"
_train_dir = f"{_repo_dir}/sampled_train"
_test_dir = f"{_repo_dir}/sampled_test"
_required_paths = [_metadata_path, _train_dir, _test_dir]

if not all(os.path.exists(p) for p in _required_paths):
    if os.path.isdir(_repo_dir):
        shutil.rmtree(_repo_dir)
    os.makedirs("./data", exist_ok=True)
    _zip_url = "https://github.com/Vicomtech/hate-speech-dataset/archive/refs/heads/master.zip"
    _zip_path = "./data/hate-speech-dataset.zip"
    urllib.request.urlretrieve(_zip_url, _zip_path)
    with zipfile.ZipFile(_zip_path, "r") as zf:
        zf.extractall("./data/")
    os.rename("./data/hate-speech-dataset-master", _repo_dir)
    os.remove(_zip_path)

_metadata = pd.read_csv(_metadata_path).set_index("file_id")

def _load_vicomtech_split(split_dir):
    rows = []
    for fname in sorted(os.listdir(split_dir)):
        if not fname.endswith(".txt"):
            continue
        file_id = fname[:-4]
        if file_id not in _metadata.index:
            continue
        label_str = _metadata.loc[file_id, "label"]
        if label_str not in ("hate", "noHate"):
            continue
        with open(os.path.join(split_dir, fname), encoding="utf-8") as f:
            text = f.read().strip()
        rows.append({"text": text, "label": 1 if label_str == "hate" else 0})
    return Dataset.from_list(rows)

vicomtech_train = _load_vicomtech_split(_train_dir)
vicomtech_test  = _load_vicomtech_split(_test_dir)

print("Vicomtech")
print(f"  Train: {len(vicomtech_train):,}  Test: {len(vicomtech_test):,}")

Vicomtech
  Train: 1,914  Test: 478


## 2. Tokenization

Transformer models cannot work directly on raw text — they require a **tokenizer** that converts text into integer token IDs and an attention mask.

We truncate and pad every sequence to a fixed length of 128 tokens. Most tweets are well under this limit, so little information is lost. Each model has its own vocabulary and tokenizer, so we re-tokenize for each model.

In [5]:
def tokenize(ds, tokenizer, text_col="post", max_length=128):
    def _tok(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    return ds.map(_tok, batched=True)

## 3. Metrics

The `Trainer` calls `compute_metrics` after each evaluation epoch. We use **macro F1** as the primary metric — it averages F1 across both classes, penalising models that ignore the minority class (hate speech). Precision and recall give additional diagnostic information.

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
        "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
        "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
    }

## 4. Fine-tuning Loop

### What is a classification head?

A pretrained model like BERT has learned rich representations of text but has no built-in notion of hate speech. `AutoModelForSequenceClassification` adds a small **linear layer** (the classification head) on top of the `[CLS]` token representation, projecting it to 2 output logits (Non-HS, HS). This head is randomly initialised.

### What does fine-tuning do?

Fine-tuning runs gradient descent on the labelled IHC training set, updating **all** model weights — both the pretrained transformer layers and the new classification head. The pretrained weights are a warm start: the model has already learned grammar, semantics, and world knowledge, so it only needs a few epochs to adapt to the new task. Training from scratch would require far more data and compute.

In [7]:
MODELS = {
    "bert-base-uncased": "bert-base-uncased",
    "hateBERT":          "GroNLP/hateBERT",
    "roberta-base":      "roberta-base",
}

DATASETS = {
    "IHC":       {"train": train_ds,        "test": test_ds,        "text_col": "post"},
    "ISHate":    {"train": ishate_train,     "test": ishate_test,    "text_col": "text"},
    "Vicomtech": {"train": vicomtech_train,  "test": vicomtech_test, "text_col": "text"},
}

results = {}

for dataset_name, dataset in DATASETS.items():
    results[dataset_name] = {}
    for model_name, model_id in MODELS.items():
        print(f"\n{'='*60}")
        print(f"Dataset: {dataset_name}  |  Model: {model_name}")
        print(f"{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

        tok_train = tokenize(dataset["train"], tokenizer, text_col=dataset["text_col"])
        tok_test  = tokenize(dataset["test"],  tokenizer, text_col=dataset["text_col"])

        training_args = TrainingArguments(
            output_dir=f"./checkpoints/{dataset_name}/{model_name}",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",
            report_to="none",
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        # Save fine-tuned weights in HuggingFace format for use in rag_step3_index.ipynb
        save_path = f"./weights/{model_name}_{dataset_name}"
        os.makedirs(save_path, exist_ok=True)
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  Saved weights → {save_path}")

        preds_output = trainer.predict(tok_test)
        preds  = np.argmax(preds_output.predictions, axis=-1)
        labels = dataset["test"]["label"]

        print(classification_report(labels, preds, target_names=["Non-HS", "HS"]))

        results[dataset_name][model_name] = {
            "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
            "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
            "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
        }


Dataset: IHC  |  Model: bert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1914 [00:00<?, ? examples/s]

Map:   0%|          | 0/478 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Macro F1,Macro P,Macro R
1,0.531100,0.465078,0.803178,0.804392,0.803347
2,0.321600,0.498047,0.809523,0.810281,0.809623
3,0.209800,0.564547,0.805432,0.805487,0.805439


  Saved weights → ./weights/roberta-base_Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.81      0.80      0.80       239
          HS       0.80      0.81      0.81       239

    accuracy                           0.81       478
   macro avg       0.81      0.81      0.81       478
weighted avg       0.81      0.81      0.81       478



## 5. Results

One table per dataset shows macro F1/P/R for all three models. Comparing across datasets reveals whether model rankings are consistent or dataset-dependent — e.g., whether HateBERT's domain-specific pretraining helps more on ISHate's subtler examples than on IHC.

In [8]:
import pandas as pd

for dataset_name, dataset_results in results.items():
    df = pd.DataFrame(dataset_results).T
    df.columns = ["Macro F1", "Macro Precision", "Macro Recall"]
    df.index.name = "Model"
    styled = df.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold; background-color: #d4f1d4").set_caption(dataset_name)
    display(styled)

,Macro F1,Macro Precision,Macro Recall
Model,,,
bert-base-uncased,0.779,0.783,0.776
hateBERT,0.782,0.790,0.777
roberta-base,0.790,0.795,0.786


,Macro F1,Macro Precision,Macro Recall
Model,,,
bert-base-uncased,0.874,0.873,0.874
hateBERT,0.871,0.871,0.871
roberta-base,0.886,0.883,0.888


,Macro F1,Macro Precision,Macro Recall
Model,,,
bert-base-uncased,0.803,0.804,0.803
hateBERT,0.824,0.824,0.824
roberta-base,0.805,0.805,0.805
